In [17]:
# Importação de bibliotecas
import os
import pandas as pd

In [ ]:
# Definição de variáveis atreladas aos caminhos
BASE_DIR = "../Fake.br-Corpus/" 

FULL_DIR_FAKE = os.path.join(BASE_DIR, "full_texts", "fake")
FULL_DIR_TRUE = os.path.join(BASE_DIR, "full_texts", "true")
TRUNC_DIR_FAKE = os.path.join(BASE_DIR, "size_normalized_texts", "fake")
TRUNC_DIR_TRUE = os.path.join(BASE_DIR, "size_normalized_texts", "true")
META_DIR_FAKE = os.path.join(BASE_DIR, "full_texts", "fake-meta-information")
META_DIR_TRUE = os.path.join(BASE_DIR, "full_texts", "true-meta-information")

# Definição estrita das 25 colunas conforme a documentação do corpus
NOME_METADADOS = [
    "author", "link", "category", "date_of_publication", 
    "number_of_tokens", "number_of_words_without_punctuation", 
    "number_of_types", "number_of_links_inside_the_news", 
    "number_of_words_in_upper_case", "number_of_verbs", 
    "number_of_subjuntive_and_imperative_verbs", "number_of_nouns", 
    "number_of_adjectives", "number_of_adverbs", "number_of_modal_verbs", 
    "number_of_singular_first_and_second_personal_pronouns", 
    "number_of_plural_first_personal_pronouns", "number_of_pronouns", 
    "pausality", "number_of_characters", "average_sentence_length", 
    "average_word_length", "percentage_of_news_with_spelling_errors", 
    "emotiveness", "diversity"
]

# Funções de leitura
def carregar_textos(caminho_diretorio):
    dados = {}
    if not os.path.exists(caminho_diretorio):
        print(f"Diretório não encontrado: {caminho_diretorio}")
        return dados
    for nome_arquivo in os.listdir(caminho_diretorio):
        if nome_arquivo.endswith(".txt"):
            id_doc = nome_arquivo.replace(".txt", "")
            with open(os.path.join(caminho_diretorio, nome_arquivo), 'r', encoding='utf-8') as f:
                dados[id_doc] = f.read().strip()
    return dados

def carregar_metadados_mapeados(caminho_diretorio):
    dados_meta = {}
    if not os.path.exists(caminho_diretorio):
        print(f"Diretório de metadados não encontrado: {caminho_diretorio}")
        return dados_meta
        
    for nome_arquivo in os.listdir(caminho_diretorio):
        if nome_arquivo.endswith(".txt"):
            # Remove o sufixo '-meta' (se houver) e o '.txt' para isolar apenas o ID numérico
            id_doc = nome_arquivo.replace("-meta.txt", "").replace(".txt", "")
            
            with open(os.path.join(caminho_diretorio, nome_arquivo), 'r', encoding='utf-8') as f:
                linhas = [linha.strip() for linha in f.readlines()]
                
                # Associa a linha do txt ao nome correto da variável
                dict_atributos = {}
                for i, nome_coluna in enumerate(NOME_METADADOS):
                    dict_atributos[nome_coluna] = linhas[i] if i < len(linhas) else None
                
                dados_meta[id_doc] = dict_atributos
                
    return dados_meta

# Carregamento dos dados
print("Carregando arquivos...")
textos_completos_fake = carregar_textos(FULL_DIR_FAKE)
textos_completos_true = carregar_textos(FULL_DIR_TRUE)
textos_trunc_fake = carregar_textos(TRUNC_DIR_FAKE)
textos_trunc_true = carregar_textos(TRUNC_DIR_TRUE)
meta_fake = carregar_metadados_mapeados(META_DIR_FAKE)
meta_true = carregar_metadados_mapeados(META_DIR_TRUE)

# Trava de Segurança
if not textos_completos_fake and not textos_completos_true:
    raise ValueError("Nenhum texto foi carregado. Verificar se o caminho no BASE_DIR está correto.")

# Estruturação no formato tabular consolidado
registros = []

def processar_base(textos_dict, trunc_dict, meta_dict, classe):
    for id_doc, texto in textos_dict.items():
        linha = {
            "id_noticia": id_doc,
            "classe": classe,
            "texto_completo": texto,
            "texto_truncado": trunc_dict.get(id_doc, "")
        }
        
        # Puxa o dicionário de atributos daquela notícia (ou cria um vazio com Nones se falhar)
        metadados = meta_dict.get(id_doc, {col: None for col in NOME_METADADOS})
        linha.update(metadados)
        
        registros.append(linha)

processar_base(textos_completos_fake, textos_trunc_fake, meta_fake, "fake")
processar_base(textos_completos_true, textos_trunc_true, meta_true, "true")

# Criação do DataFrame e exportação
df_corpus = pd.DataFrame(registros)

# Converte IDs para numérico para ordenar adequadamente
df_corpus['id_noticia'] = pd.to_numeric(df_corpus['id_noticia'])
df_corpus = df_corpus.sort_values(by=['classe', 'id_noticia']).reset_index(drop=True)

df_corpus.to_csv('fake_br_estruturado.csv', index=False, encoding='utf-8')
print(f"\nSucesso! Base gerada com {len(df_corpus)} registros e as colunas originais do corpus. Salvo como 'fake_br_estruturado.csv'.")

Carregando arquivos...

Sucesso! Base gerada com 7200 registros e as colunas originais do corpus. Salvo como 'fake_br_estruturado.csv'.


In [16]:
# Define o nome do arquivo gerado no passo anterior
caminho_csv = 'fake_br_estruturado.csv'

# Lê o CSV e armazena no DataFrame
df_fake_br = pd.read_csv(caminho_csv)

print("Sucesso! DataFrame salvo como 'df_fake_br'.")

Sucesso! DataFrame salvo como 'df_fake_br'.


In [21]:
# Exibe as 5 primeiras linhas para confirmar que carregou corretamente
print("Visualização das primeiras linhas:")
df_fake_br.head()

Visualização das primeiras linhas:


,id_noticia,classe,texto_completo,texto_truncado,author,link,category,date_of_publication,number_of_tokens,number_of_words_without_punctuation,...,number_of_singular_first_and_second_personal_pronouns,number_of_plural_first_personal_pronouns,number_of_pronouns,pausality,number_of_characters,average_sentence_length,average_word_length,percentage_of_news_with_spelling_errors,emotiveness,diversity
0,1,fake,Kátia Abreu diz que vai colocar sua expulsão e...,Kátia Abreu diz que vai colocar sua expulsão e...,mrk,https://ceticismopolitico.com/2017/11/30/katia...,politica,2017-11-30,211,185,...,2,0,26,2.00000,815,14.2308,4.40541,0.000000,0.263158,0.648649
1,2,fake,Blog esquerdista dá a entender que reclamar de...,Blog esquerdista dá a entender que reclamar de...,NaN,https://ceticismopolitico.com/2017/11/28/blog-...,sociedade_cotidiano,2017-11-28,356,300,...,0,0,30,3.29412,1321,17.6471,4.40333,0.013333,0.277372,0.623333
2,3,fake,"Alckmin diz que por ele PSDB desembarca, mas...","Alckmin diz que por ele PSDB “desembarca”, mas...",NaN,https://ceticismopolitico.com/2017/11/28/alckm...,politica,2017-11-28,274,224,...,1,0,14,3.57143,1075,16.0000,4.79911,0.000000,0.262136,0.669643
3,4,fake,Cara de pau não tem limites: Zé Celso aciona M...,Cara de pau não tem limites: Zé Celso aciona M...,NaN,https://ceticismopolitico.com/2017/11/28/cara-...,politica,2017-11-28,319,291,...,0,0,23,2.33333,1341,24.2500,4.60825,0.003436,0.182540,0.635739
4,5,fake,Temer resolve o problema de Luislinda: liberd...,Temer resolve o problema de Luislinda: “liberd...,NaN,https://ceticismopolitico.com/2017/11/27/temer...,politica,2017-11-27,162,133,...,0,0,12,2.63636,654,12.0909,4.91729,0.000000,0.303571,0.721804
